In [ ]:
import math
from decimal import Decimal, getcontext

class DecimalCustomFloat:
    def __init__(self, value, mantissa_digits=4):
        """
        Initialize the DecimalCustomFloat with specified mantissa digit length.

        Parameters:
        - value (float or Decimal): The number to represent.
        - mantissa_digits (int): Number of digits in the mantissa.
        """
        self.mantissa_digits = mantissa_digits
        self.value = Decimal(str(value))
        self.sign = '-' if self.value < 0 else '+'
        self.exponent = 0
        self.mantissa = Decimal('0')

        if self.value.is_zero():
            self.exponent = 0
            self.mantissa = Decimal('0')
        else:
            self._normalize()

    def _normalize(self):
        # Set precision for Decimal operations
        getcontext().prec = self.mantissa_digits + 5  # Extra precision

        abs_value = self.value.copy_abs()

        # Determine exponent b
        self.exponent = abs_value.adjusted() + 1

        # Compute unrounded mantissa
        mantissa = abs_value.scaleb(-self.exponent)

        # Get mantissa digits up to m+1 places
        mantissa_str = format(mantissa, 'f')
        if '.' in mantissa_str:
            _, fractional_part = mantissa_str.split('.')
        else:
            fractional_part = ''

        # Ensure enough digits
        fractional_part = fractional_part.ljust(self.mantissa_digits + 1, '0')
        mantissa_digits_str = fractional_part[:self.mantissa_digits + 1]

        # Extract digits
        mantissa_m = mantissa_digits_str[:self.mantissa_digits]
        digit_m_plus_1 = int(mantissa_digits_str[self.mantissa_digits])

        # Create mantissa
        mantissa_rounded = Decimal('0.' + mantissa_m)

        # Apply rounding
        if digit_m_plus_1 >= 5:
            increment = Decimal('1e-' + str(self.mantissa_digits))
            mantissa_rounded += increment

        # Handle mantissa overflow
        if mantissa_rounded >= Decimal('1'):
            mantissa_rounded = Decimal('0.1')
            self.exponent += 1

        self.mantissa = mantissa_rounded.normalize()

    def __repr__(self):
        return f"{self.sign}{self.mantissa} * 10^{self.exponent}"

    def to_decimal(self):
        """
        Convert the DecimalCustomFloat back to a Decimal number.
        """
        sign = -1 if self.sign == '-' else 1
        return sign * self.mantissa * (Decimal('10') ** self.exponent)

    # Arithmetic Operations with Rounding
    def _operate(self, other, operation):
        if isinstance(other, DecimalCustomFloat):
            value_other = other.to_decimal()
            mantissa_digits = min(self.mantissa_digits, other.mantissa_digits)
        else:
            value_other = Decimal(str(other))
            mantissa_digits = self.mantissa_digits

        # Perform the operation
        getcontext().prec = mantissa_digits + 5  # Extra precision
        result_value = operation(self.to_decimal(), value_other)

        # Round the result using the same mantissa_digits
        return DecimalCustomFloat(result_value, mantissa_digits)

    def __add__(self, other):
        return self._operate(other, lambda x, y: x + y)

    def __sub__(self, other):
        return self._operate(other, lambda x, y: x - y)

    def __mul__(self, other):
        return self._operate(other, lambda x, y: x * y)

    def __truediv__(self, other):
        return self._operate(other, lambda x, y: x / y)

    # Reverse operations to support operations where DecimalCustomFloat is on the right
    def __radd__(self, other):
        return self.__add__(other)

    def __rsub__(self, other):
        return DecimalCustomFloat(other, self.mantissa_digits).__sub__(self)

    def __rmul__(self, other):
        return self.__mul__(other)

    def __rtruediv__(self, other):
        return DecimalCustomFloat(other, self.mantissa_digits).__truediv__(self)


class CustomFloat:
    def __init__(self, value, mantissa_bits=23):
        """
        Initialize the CustomFloat with a specified mantissa length.

        Parameters:
        - value (float): The floating-point number to represent.
        - mantissa_bits (int): Number of bits in the mantissa.
        """
        self.mantissa_bits = mantissa_bits
        self.value = float(value)
        self.sign, self.exponent, self.mantissa = self.float_to_components(self.value)
    
    def float_to_components(self, value):
        """
        Decompose the float into sign, exponent, and mantissa.

        Returns:
        - sign (int): 0 for positive, 1 for negative.
        - exponent (int): Exponent value.
        - mantissa (float): Mantissa value.
        """
        if value == 0.0:
            return 0, 0, 0.0
        
        sign = 0
        if value < 0:
            sign = 1
            value = -value
        
        exponent = math.floor(math.log2(value))
        mantissa = value / (2 ** exponent) - 1  # Normalize mantissa to [0, 1)
        
        return sign, exponent, mantissa
    
    def components_to_float(self, sign, exponent, mantissa):
        """
        Reconstruct the float from sign, exponent, and mantissa.

        Returns:
        - value (float): The reconstructed floating-point number.
        """
        return ((-1) ** sign) * (1 + mantissa) * (2 ** exponent)
    
    def quantize_mantissa(self):
        """
        Quantize the mantissa to the specified number of bits.

        Returns:
        - quantized_mantissa (float): The mantissa after quantization.
        """
        quantization_step = 1 / (2 ** self.mantissa_bits)
        quantized_mantissa = math.floor(self.mantissa / quantization_step) * quantization_step
        return quantized_mantissa
    
    def to_custom_float(self):
        """
        Convert the internal components to the quantized custom float.

        Returns:
        - custom_value (float): The custom floating-point number with specified mantissa bits.
        """
        quantized_mantissa = self.quantize_mantissa()
        return self.components_to_float(self.sign, self.exponent, quantized_mantissa)
    
    def __repr__(self):
        custom_value = self.to_custom_float()
        return f"CustomFloat(value={custom_value}, mantissa_bits={self.mantissa_bits})"
    
    # Arithmetic Operations
    def __add__(self, other):
        if isinstance(other, CustomFloat):
            result = self.to_custom_float() + other.to_custom_float()
        else:
            result = self.to_custom_float() + other
        return CustomFloat(result, self.mantissa_bits)
    
    def __sub__(self, other):
        if isinstance(other, CustomFloat):
            result = self.to_custom_float() - other.to_custom_float()
        else:
            result = self.to_custom_float() - other
        return CustomFloat(result, self.mantissa_bits)
    
    def __mul__(self, other):
        if isinstance(other, CustomFloat):
            result = self.to_custom_float() * other.to_custom_float()
        else:
            result = self.to_custom_float() * other
        return CustomFloat(result, self.mantissa_bits)
    
    def __truediv__(self, other):
        if isinstance(other, CustomFloat):
            result = self.to_custom_float() / other.to_custom_float()
        else:
            result = self.to_custom_float() / other
        return CustomFloat(result, self.mantissa_bits)
    
    # Comparison Operations
    def __gt__(self, other):
        """
        Greater-than comparison.

        Returns:
        - bool: True if self > other, False otherwise.
        """
        if isinstance(other, CustomFloat):
            return self.to_custom_float() > other.to_custom_float()
        else:
            return self.to_custom_float() > other
    
    def __lt__(self, other):
        """
        Less-than comparison.

        Returns:
        - bool: True if self < other, False otherwise.
        """
        if isinstance(other, CustomFloat):
            return self.to_custom_float() < other.to_custom_float()
        else:
            return self.to_custom_float() < other
    
    def __eq__(self, other):
        """
        Equality comparison.

        Returns:
        - bool: True if self == other, False otherwise.
        """
        if isinstance(other, CustomFloat):
            return self.to_custom_float() == other.to_custom_float()
        else:
            return self.to_custom_float() == other
    
    def __ge__(self, other):
        """
        Greater-than or equal comparison.

        Returns:
        - bool: True if self >= other, False otherwise.
        """
        return self > other or self == other
    
    def __le__(self, other):
        """
        Less-than or equal comparison.

        Returns:
        - bool: True if self <= other, False otherwise.
        """
        return self < other or self == other
    
    def __ne__(self, other):
        """
        Not equal comparison.

        Returns:
        - bool: True if self != other, False otherwise.
        """
        return not self == other

In [ ]:
# Testing with zero
g = DecimalCustomFloat('0', mantissa_digits=4)
h = a + g
print(f"a + 0: {h}")  # Output: +0.3142 * 10^1
print(f"(a + 0) as decimal: {h.to_decimal()}")  # Output: 3.142

NameError: name 'a' is not defined

In [ ]:
# Create instances
a = DecimalCustomFloat('3.1415926', mantissa_digits=4)
b = DecimalCustomFloat('2.7182818', mantissa_digits=4)

print(f"a: {a}")  # Output: +0.3142 * 10^1
print(f"b: {b}")  # Output: +0.2718 * 10^1

# Addition
c = a + b
print(f"a + b: {c}")  # Output: +0.5860 * 10^1
print(f"(a + b) as decimal: {c.to_decimal()}")  # Output: 5.860

# Subtraction
d = a - b
print(f"a - b: {d}")  # Output: +0.0424 * 10^1
print(f"(a - b) as decimal: {d.to_decimal()}")  # Output: 0.424

# Multiplication
e = a * b
print(f"a * b: {e}")  # Output: +0.8535 * 10^1
print(f"(a * b) as decimal: {e.to_decimal()}")  # Output: 8.535

# Division
f = a / b
print(f"a / b: {f}")  # Output: +0.1156 * 10^1
print(f"(a / b) as decimal: {f.to_decimal()}")  # Output: 1.156

a: +0.3142 * 10^1
b: +0.2718 * 10^1
a + b: +0.586 * 10^1
(a + b) as decimal: 5.860
a - b: +0.424 * 10^0
(a - b) as decimal: 0.424
a * b: +0.854 * 10^1
(a * b) as decimal: 8.540
a / b: +0.1156 * 10^1
(a / b) as decimal: 1.1560


In [ ]:
# Create instances
a = DecimalCustomFloat('0.75', mantissa_digits=2)
b = DecimalCustomFloat('0.055', mantissa_digits=2)
c = DecimalCustomFloat('0.8', mantissa_digits=2)

In [ ]:
test = a + b - c

In [ ]:
print(a, b, c, test)

+0.75 * 10^0 +0.55 * 10^-1 +0.8 * 10^0 +0.1 * 10^-1


In [ ]:
print(a)

+0.75 * 10^0


In [ ]:
print(b)

+0.55 * 10^-1


In [ ]:
print(c)

+0.8 * 10^0


In [ ]:
print(test)

+0.1 * 10^-1


In [ ]:
test2 = (b + c) + a

In [ ]:
print(test2)

+0.16 * 10^1


In [ ]:
test2 = (b - c) + a

In [ ]:
print(test2)

+0 * 10^0


In [ ]:
# Create instances
a = DecimalCustomFloat('0.75', mantissa_digits=23)
b = DecimalCustomFloat('0.055', mantissa_digits=23)
c = DecimalCustomFloat('0.8', mantissa_digits=23)

In [ ]:
test = a + b - c

In [ ]:
test2 = (b - c) + a

In [ ]:
print(test)

+0.5 * 10^-2


In [ ]:
print(test2)

+0.5 * 10^-2


In [ ]:
import math
from decimal import Decimal, getcontext
import math

class DecimalCustomFloat:
    def __init__(self, value, mantissa_digits=4):
        """
        Initialize the DecimalCustomFloat with specified mantissa digit length.

        Parameters:
        - value (float or Decimal): The number to represent.
        - mantissa_digits (int): Number of digits in the mantissa.
        """
        self.mantissa_digits = mantissa_digits
        self.value = Decimal(str(value))
        self.sign = '-' if self.value < 0 else '+'
        self.exponent = 0
        self.mantissa = Decimal('0')

        if self.value.is_zero():
            self.exponent = 0
            self.mantissa = Decimal('0')
        else:
            self._normalize()

    def _normalize(self):
        # Set precision for Decimal operations
        getcontext().prec = self.mantissa_digits + 5  # Extra precision

        abs_value = self.value.copy_abs()

        # Determine exponent b
        self.exponent = abs_value.adjusted() + 1

        # Compute unrounded mantissa
        mantissa = abs_value.scaleb(-self.exponent)

        # Get mantissa digits up to m+1 places
        mantissa_str = format(mantissa, 'f')
        if '.' in mantissa_str:
            _, fractional_part = mantissa_str.split('.')
        else:
            fractional_part = ''

        # Ensure enough digits
        fractional_part = fractional_part.ljust(self.mantissa_digits + 1, '0')
        mantissa_digits_str = fractional_part[:self.mantissa_digits + 1]

        # Extract digits
        mantissa_m = mantissa_digits_str[:self.mantissa_digits]
        digit_m_plus_1 = int(mantissa_digits_str[self.mantissa_digits])

        # Create mantissa
        mantissa_rounded = Decimal('0.' + mantissa_m)

        # Apply rounding
        if digit_m_plus_1 >= 5:
            increment = Decimal('1e-' + str(self.mantissa_digits))
            mantissa_rounded += increment

        # Handle mantissa overflow
        if mantissa_rounded >= Decimal('1'):
            mantissa_rounded = Decimal('0.1')
            self.exponent += 1

        self.mantissa = mantissa_rounded.normalize()

    def __repr__(self):
        return f"{self.sign}{self.mantissa} * 10^{self.exponent}"

    def to_decimal(self):
        """
        Convert the DecimalCustomFloat back to a Decimal number.
        """
        sign = -1 if self.sign == '-' else 1
        return sign * self.mantissa * (Decimal('10') ** self.exponent)

    # Arithmetic Operations with Rounding
    def _operate(self, other, operation):
        if isinstance(other, DecimalCustomFloat):
            value_other = other.to_decimal()
            mantissa_digits = min(self.mantissa_digits, other.mantissa_digits)
        else:
            value_other = Decimal(str(other))
            mantissa_digits = self.mantissa_digits

        # Perform the operation
        getcontext().prec = mantissa_digits + 5  # Extra precision
        result_value = operation(self.to_decimal(), value_other)

        # Round the result using the same mantissa_digits
        return DecimalCustomFloat(result_value, mantissa_digits)

    def __add__(self, other):
        return self._operate(other, lambda x, y: x + y)

    def __sub__(self, other):
        return self._operate(other, lambda x, y: x - y)

    def __mul__(self, other):
        return self._operate(other, lambda x, y: x * y)

    def __truediv__(self, other):
        return self._operate(other, lambda x, y: x / y)

    # Reverse operations to support operations where DecimalCustomFloat is on the right
    def __radd__(self, other):
        return self.__add__(other)

    def __rsub__(self, other):
        return DecimalCustomFloat(other, self.mantissa_digits).__sub__(self)

    def __rmul__(self, other):
        return self.__mul__(other)

    def __rtruediv__(self, other):
        return DecimalCustomFloat(other, self.mantissa_digits).__truediv__(self)

    # Comparison Operations
    def __lt__(self, other):
        if isinstance(other, DecimalCustomFloat):
            return self.to_decimal() < other.to_decimal()
        else:
            return self.to_decimal() < Decimal(str(other))

    def __gt__(self, other):
        if isinstance(other, DecimalCustomFloat):
            return self.to_decimal() > other.to_decimal()
        else:
            return self.to_decimal() > Decimal(str(other))

    def __eq__(self, other):
        if isinstance(other, DecimalCustomFloat):
            return self.to_decimal() == other.to_decimal()
        else:
            return self.to_decimal() == Decimal(str(other))

    def __le__(self, other):
        return self < other or self == other

    def __ge__(self, other):
        return self > other or self == other

    def __ne__(self, other):
        return not self == other


class CustomFloat:
    def __init__(self, value, mantissa_bits=23):
        """
        Initialize the CustomFloat with a specified mantissa length.

        Parameters:
        - value (float): The floating-point number to represent.
        - mantissa_bits (int): Number of bits in the mantissa.
        """
        self.mantissa_bits = mantissa_bits
        self.value = float(value)
        self.sign, self.exponent, self.mantissa = self.float_to_components(self.value)
    
    def float_to_components(self, value):
        """
        Decompose the float into sign, exponent, and mantissa.

        Returns:
        - sign (int): 0 for positive, 1 for negative.
        - exponent (int): Exponent value.
        - mantissa (float): Mantissa value.
        """
        if value == 0.0:
            return 0, 0, 0.0
        
        sign = 0
        if value < 0:
            sign = 1
            value = -value
        
        exponent = math.floor(math.log2(value))
        mantissa = value / (2 ** exponent) - 1  # Normalize mantissa to [0, 1)
        
        return sign, exponent, mantissa
    
    def components_to_float(self, sign, exponent, mantissa):
        """
        Reconstruct the float from sign, exponent, and mantissa.

        Returns:
        - value (float): The reconstructed floating-point number.
        """
        return ((-1) ** sign) * (1 + mantissa) * (2 ** exponent)
    
    def quantize_mantissa(self):
        """
        Quantize the mantissa to the specified number of bits.

        Returns:
        - quantized_mantissa (float): The mantissa after quantization.
        """
        quantization_step = 1 / (2 ** self.mantissa_bits)
        quantized_mantissa = math.floor(self.mantissa / quantization_step) * quantization_step
        return quantized_mantissa
    
    def to_custom_float(self):
        """
        Convert the internal components to the quantized custom float.

        Returns:
        - custom_value (float): The custom floating-point number with specified mantissa bits.
        """
        quantized_mantissa = self.quantize_mantissa()
        return self.components_to_float(self.sign, self.exponent, quantized_mantissa)
    
    def __repr__(self):
        custom_value = self.to_custom_float()
        return f"CustomFloat(value={custom_value}, mantissa_bits={self.mantissa_bits})"
    
    # Arithmetic Operations
    def __add__(self, other):
        if isinstance(other, CustomFloat):
            result = self.to_custom_float() + other.to_custom_float()
        else:
            result = self.to_custom_float() + other
        return CustomFloat(result, self.mantissa_bits)
    
    def __sub__(self, other):
        if isinstance(other, CustomFloat):
            result = self.to_custom_float() - other.to_custom_float()
        else:
            result = self.to_custom_float() - other
        return CustomFloat(result, self.mantissa_bits)
    
    def __mul__(self, other):
        if isinstance(other, CustomFloat):
            result = self.to_custom_float() * other.to_custom_float()
        else:
            result = self.to_custom_float() * other
        return CustomFloat(result, self.mantissa_bits)
    
    def __truediv__(self, other):
        if isinstance(other, CustomFloat):
            result = self.to_custom_float() / other.to_custom_float()
        else:
            result = self.to_custom_float() / other
        return CustomFloat(result, self.mantissa_bits)
    
    # Comparison Operations
    def __gt__(self, other):
        """
        Greater-than comparison.

        Returns:
        - bool: True if self > other, False otherwise.
        """
        if isinstance(other, CustomFloat):
            return self.to_custom_float() > other.to_custom_float()
        else:
            return self.to_custom_float() > other
    
    def __lt__(self, other):
        """
        Less-than comparison.

        Returns:
        - bool: True if self < other, False otherwise.
        """
        if isinstance(other, CustomFloat):
            return self.to_custom_float() < other.to_custom_float()
        else:
            return self.to_custom_float() < other
    
    def __eq__(self, other):
        """
        Equality comparison.

        Returns:
        - bool: True if self == other, False otherwise.
        """
        if isinstance(other, CustomFloat):
            return self.to_custom_float() == other.to_custom_float()
        else:
            return self.to_custom_float() == other
    
    def __ge__(self, other):
        """
        Greater-than or equal comparison.

        Returns:
        - bool: True if self >= other, False otherwise.
        """
        return self > other or self == other
    
    def __le__(self, other):
        """
        Less-than or equal comparison.

        Returns:
        - bool: True if self <= other, False otherwise.
        """
        return self < other or self == other
    
    def __ne__(self, other):
        """
        Not equal comparison.

        Returns:
        - bool: True if self != other, False otherwise.
        """
        return not self == other

In [ ]:
# Create instances
a = DecimalCustomFloat('0.75', mantissa_digits=4)
b = DecimalCustomFloat('0.055', mantissa_digits=4)
c = DecimalCustomFloat('0.8', mantissa_digits=4)

In [ ]:
# Create instances
a = DecimalCustomFloat('0.75', mantissa_digits=2)
b = DecimalCustomFloat('0.055', mantissa_digits=2)
c = DecimalCustomFloat('0.8', mantissa_digits=2)

In [ ]:
test1 = (a + b) - c

In [ ]:
print(test1)

+0.1 * 10^-1


In [ ]:
test2 = a + (b - c)

In [ ]:
print(test2)

+0 * 10^0


In [ ]:
def compute_machine_accuracy(mantissa_digits):
    one = DecimalCustomFloat(1, mantissa_digits)
    epsilon = DecimalCustomFloat(1, mantissa_digits)
    half = DecimalCustomFloat(0.5, mantissa_digits)

    while one + epsilon > one:
        epsilon = epsilon * half

    machine_epsilon = epsilon * DecimalCustomFloat(2, mantissa_digits)
    return machine_epsilon.to_decimal()

In [ ]:
def compute_machine_accuracy(mantissa_digits=23):
    one = DecimalCustomFloat(1, mantissa_digits)
    epsilon = DecimalCustomFloat(1, mantissa_digits)
    half = DecimalCustomFloat(0.5, mantissa_digits)

    while one + epsilon > one:
        epsilon = epsilon * half

    machine_epsilon = epsilon * DecimalCustomFloat(2, mantissa_digits)
    return machine_epsilon.to_decimal()

In [ ]:
compute_machine_accuracy()

Decimal('5.2939559203393771191796E-23')

In [ ]:
def compute_machine_accuracy(mantissa_digits=21):
    one = DecimalCustomFloat(1, mantissa_digits)
    epsilon = DecimalCustomFloat(1, mantissa_digits)
    half = DecimalCustomFloat(0.5, mantissa_digits)

    while one + epsilon > one:
        epsilon = epsilon * half

    machine_epsilon = epsilon * DecimalCustomFloat(2, mantissa_digits)
    return machine_epsilon.to_decimal()

In [ ]:
compute_machine_accuracy()

Decimal('6.77626357803440271286E-21')

In [ ]:
def compute_machine_accuracy(mantissa_digits=23):
    one = DecimalCustomFloat(1, mantissa_digits)
    epsilon = DecimalCustomFloat(1, mantissa_digits)
    half = DecimalCustomFloat(0.5, mantissa_digits)

    while one + epsilon > one:
        epsilon = epsilon * half

    machine_epsilon = epsilon * DecimalCustomFloat(2, mantissa_digits)
    return machine_epsilon.to_decimal()

In [ ]:
compute_machine_accuracy(4)

Decimal('0.000977')

In [ ]:
compute_machine_accuracy(2)

Decimal('0.066')

In [ ]:
compute_machine_accuracy(12)

Decimal('7.27595761436E-12')

In [ ]:
compute_machine_accuracy(13)

Decimal('9.09494701775E-13')

In [ ]:
compute_machine_accuracy(14)

Decimal('5.684341886082E-14')

In [ ]:
def compute_machine_accuracy(mantissa_digits=4):
    one = DecimalCustomFloat(1, mantissa_digits)
    epsilon = DecimalCustomFloat(1, mantissa_digits)
    half = DecimalCustomFloat(0.5, mantissa_digits)
    two = DecimalCustomFloat(2, mantissa_digits)

    while one + epsilon > one:
        epsilon = epsilon * half

    machine_epsilon = epsilon * two
    return machine_epsilon.to_decimal()

In [ ]:
compute_machine_accuracy(7)

Decimal('9.53675E-7')

In [ ]:
compute_machine_accuracy(2)

Decimal('0.066')